# LLM Zoomcamp 2026 - Homework 1: Agentic RAG by Jose Castro

This notebook contains my run for Homework 1 of the DataTalksClub LLM Zoomcamp 2026 course.

Official homework: [`cohorts/2026/01-agentic-rag/homework.md`](https://github.com/DataTalksClub/llm-zoomcamp/blob/main/cohorts/2026/01-agentic-rag/homework.md)

The task is to build a small RAG pipeline over the course lesson files and then turn it into a simple agentic loop. I used the repository at commit `8c1834d`, as required in the homework, so the lesson files match the reference dataset.

Main tools used here:

- `gitsource` for reading the markdown lesson files from GitHub
- `minsearch` for keyword search
- OpenAI Responses API with `gpt-5.4-mini`

The homework notes that some LLM-based answers may not match exactly, so I use the closest multiple-choice option where needed.


## Setup

I use the same basic environment as in the module. The OpenAI API key is read from the `OPENAI_API_KEY` environment variable, so the key is not stored in this notebook.


In [ ]:
# !pip install gitsource minsearch openai

import os
import json
from openai import OpenAI

client = OpenAI()

# The OpenAI API key should be available as OPENAI_API_KEY.
# I keep it outside the notebook so it is not committed to GitHub.

MODEL = "gpt-5.4-mini"


## Preparation

The homework asks us to read all lesson pages from the course repository, using commit `8c1834d`. The filter keeps only markdown files under a `lessons/` folder.


In [ ]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

documents = []
for file in files:
    doc = file.parse()
    documents.append(doc)

print(f"Downloaded files: {len(files)}")
print(f"Parsed documents: {len(documents)}")
print()
print("Sample document:")
print(documents[0]["filename"], "-", len(documents[0]["content"]), "characters")



Downloaded files: 72
Parsed documents: 72

Sample document:
01-agentic-rag/lessons/01-intro.md - 3183 characters


## Q1. How many lesson pages?

The filter is not limited to the `01-agentic-rag` module. It goes through the seven course modules and keeps every markdown file whose path contains `/lessons/`.

Options:

- 24
- 72
- 240
- 720


In [ ]:
q1_answer = len(documents)
print("Q1 - lesson pages:", q1_answer)


Q1 - lesson pages: 72


## Q2. Indexing and searching

I index the lesson documents with `minsearch`, using `content` as a text field and `filename` as a keyword field.

Query:

> How does the agentic loop keep calling the model until it stops?

Options:

- `01-agentic-rag/lessons/03-rag.md`
- `01-agentic-rag/lessons/14-agentic-loop.md`
- `04-evaluation/lessons/13-llm-as-judge.md`
- `06-best-practices/lessons/02-hybrid-search.md`


In [ ]:
from minsearch import Index

index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)
index.fit(documents)

query = "How does the agentic loop keep calling the model until it stops?"
results = index.search(query, num_results=5)

print("Top 5 results:")
for i, r in enumerate(results):
    print(i, "-", r["filename"])

q2_answer = results[0]["filename"]
print()
print("Q2 - top filename:", q2_answer)



Top 5 results:
0 - 01-agentic-rag/lessons/14-agentic-loop.md
1 - 01-agentic-rag/lessons/15-frameworks.md
2 - 01-agentic-rag/lessons/13-function-calling.md
3 - 01-agentic-rag/lessons/11-agents-intro.md
4 - 01-agentic-rag/lessons/16-other-frameworks.md

Q2 - top filename: 01-agentic-rag/lessons/14-agentic-loop.md


## Q3. RAG

The course helper `RAGBase` was written for the FAQ schema (`section`, `question`, `answer`). The lesson files use `filename` and `content`, so I adapt `search()` and `build_context()`.

I also return the full API response from `llm()` so I can read token usage.

Query:

> How does the agentic loop keep calling the model until it stops?

Options:

- 700
- 7000
- 70000
- 700000


In [ ]:
import urllib.request

url = "https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py"
urllib.request.urlretrieve(url, "rag_helper.py")

print("Downloaded rag_helper.py")


Downloaded rag_helper.py


In [ ]:
from rag_helper import RAGBase, INSTRUCTIONS, PROMPT_TEMPLATE


class LessonRAG(RAGBase):
    """RAG helper adapted to the lesson-page schema."""

    def __init__(self, index, llm_client, model=MODEL):
        self.index = index
        self.llm_client = llm_client
        self.instructions = INSTRUCTIONS
        self.prompt_template = PROMPT_TEMPLATE
        self.model = model

    def search(self, query, num_results=5):
        return self.index.search(query, num_results=num_results)

    def build_context(self, search_results):
        lines = []
        for doc in search_results:
            lines.append("FILE: " + doc["filename"])
            lines.append(doc["content"])
            lines.append("")
        return "\n".join(lines).strip()

    def llm(self, prompt):
        input_messages = [
            {"role": "developer", "content": self.instructions},
            {"role": "user", "content": prompt},
        ]

        response = self.llm_client.responses.create(
            model=self.model,
            input=input_messages,
        )

        return response

    def rag(self, query):
        search_results = self.search(query)
        prompt = self.build_prompt(query, search_results)
        response = self.llm(prompt)
        return response.output_text, response.usage


rag_lessons = LessonRAG(index=index, llm_client=client)

query = "How does the agentic loop keep calling the model until it stops?"

answer_q3, usage_q3 = rag_lessons.rag(query)

print("Model answer:")
print(answer_q3)
print()

print("Input tokens:", usage_q3.input_tokens)
print("Output tokens:", usage_q3.output_tokens)
print("Total tokens:", usage_q3.total_tokens)



Model answer:
It keeps calling the model in a `while True` loop and checks whether the model returned any `function_call` items.

- If the response includes a function call, the code runs the tool, appends the tool result to `messages`, and loops again.
- If the response has no function calls, it `break`s out of the loop.

So the stop condition is: no function calls in the latest model response.

Input tokens: 7126
Output tokens: 92
Total tokens: 7218


## Q4. Chunking

The lesson files are fairly long, so I split them into overlapping chunks with `gitsource.chunk_documents`.

Parameters:

- `size=2000`
- `step=1000`
- overlap: 1000 characters

Options:

- 70
- 295
- 1100
- 4500


In [ ]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

print("Number of chunks:", len(chunks))
print()
print("Sample chunk:")
print(chunks[0]["filename"], "| start:", chunks[0]["start"], "| len(content):", len(chunks[0]["content"]))

q4_answer = len(chunks)
print()
print("Q4 - number of chunks:", q4_answer)



Number of chunks: 295

Sample chunk:
01-agentic-rag/lessons/01-intro.md | start: 0 | len(content): 2000

Q4 - number of chunks: 295


## Q5. RAG with chunking

I index the chunks using the same fields as before, then run the same RAG query again. The goal is to compare token usage against Q3.

Options:

- about the same
- 3x fewer
- 10x fewer
- 30x fewer


In [ ]:
chunk_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)
chunk_index.fit(chunks)

rag_chunks = LessonRAG(index=chunk_index, llm_client=client)

answer_q5, usage_q5 = rag_chunks.rag(query)

print("Model answer with chunking:")
print(answer_q5)
print()
print("Usage:", usage_q5)



Model answer with chunking:
The loop keeps calling the model inside a `while True` loop. After each model response, it checks whether any `function_call` items were returned:

- If there is a function call, the code runs the tool, appends the result to `messages`, and loops again.
- If there are no function calls, `has_function_calls` stays `False`, and the loop breaks.

So the stop condition is: no function calls in that turn means the agent is done.

Usage: ResponseUsage(input_tokens=2309, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=103, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=2412)


In [ ]:
q3_input_tokens = usage_q3.input_tokens
q5_input_tokens = usage_q5.input_tokens

print("Q3 input tokens:", q3_input_tokens)
print("Q5 input tokens:", q5_input_tokens)



Q3 input tokens: 7126
Q5 input tokens: 2309


In [ ]:
ratio = q3_input_tokens / q5_input_tokens

print("Q3 input tokens without chunking:", q3_input_tokens)
print("Q5 input tokens with chunking:", q5_input_tokens)
print(f"Q3 / Q5 ratio: {ratio:.1f}x fewer tokens with chunking")



Q3 input tokens without chunking: 7126
Q5 input tokens with chunking: 2309
Q3 / Q5 ratio: 3.1x fewer tokens with chunking


## Q6. Turning it into an agent

For the agentic version, I give the model a `search` tool over the chunk index and let it decide when to call it. The homework suggests `toyaikit`, but also allows a hand-written loop. I use a manual loop here because it makes the search-call count explicit.

Agent instructions:

> You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering.

Question:

> How does the agentic loop work, and how is it different from plain RAG?

Options:

- 0
- 4
- 10
- 20

The exact count can vary between runs, so I use the closest option.


In [ ]:
call_count = 0

def search(query: str) -> str:
    """Search the course lesson chunks for content relevant to the given query.

    Args:
        query: Keywords to search for in the lesson content.
    """
    global call_count
    call_count += 1

    results = chunk_index.search(query, num_results=5)
    payload = [
        {"filename": r["filename"], "content": r["content"]}
        for r in results
    ]
    return json.dumps(payload)


tools = [
    {
        "type": "function",
        "name": "search",
        "description": "Search the course lesson chunks for content relevant to the given query.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "Keywords to search for in the lesson content.",
                }
            },
            "required": ["query"],
        },
    }
]

agent_instructions = (
    "You're a course teaching assistant. Answer the student's question using the "
    "search tool. Make multiple searches with different keywords before answering."
)

agent_question = "How does the agentic loop work, and how is it different from plain RAG?"

conversation = [
    {"role": "developer", "content": agent_instructions},
    {"role": "user", "content": agent_question},
]

MAX_ITERS = 20
final_answer = None

for _ in range(MAX_ITERS):
    response = client.responses.create(
        model=MODEL,
        input=conversation,
        tools=tools,
    )

    function_calls = [item for item in response.output if item.type == "function_call"]

    if not function_calls:
        final_answer = response.output_text
        break

    conversation += response.output

    for fc in function_calls:
        args = json.loads(fc.arguments)
        result = search(**args)
        conversation.append({
            "type": "function_call_output",
            "call_id": fc.call_id,
            "output": result,
        })

print("Final agent answer:")
print(final_answer)



Final agent answer:
The agentic loop is the pattern where the model and your code take turns:

1. Send the user request to the LLM.
2. The LLM may return a tool/function call instead of a final answer.
3. Your code executes that tool.
4. You send the tool result back to the LLM.
5. Repeat until the model returns a final answer.

Plain RAG is simpler: retrieve once, build one prompt, call the LLM once, and return the answer. The agentic loop can search more than once and decide what to do next before giving the final answer.


In [ ]:
q6_call_count = call_count
print("Q6 - search() calls:", q6_call_count)


Q6 - search() calls: 3


## Answer summary


In [ ]:
print("Q1 (lesson pages):           ", q1_answer)
print("Q2 (top filename):            ", q2_answer)
print("Q3 (input tokens, no chunks): ", q3_input_tokens)
print("Q4 (number of chunks):        ", q4_answer)
print("Q5 (Q3/Q5 token ratio):       ", f"{ratio:.1f}x")
print("Q6 (search calls):            ", q6_call_count)



Q1 (lesson pages):            72
Q2 (top filename):             01-agentic-rag/lessons/14-agentic-loop.md
Q3 (input tokens, no chunks):  7126
Q4 (number of chunks):         295
Q5 (Q3/Q5 token ratio):        3.1x
Q6 (search calls):             3
